# HP Group Analysis

Creatures with identical actual HP in the same CR tier, but different predicted HP.
The prediction spread reveals which features drive the differences — and which costs may need tuning.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/engineered_features.csv')
contributions_df = pd.read_csv('../../data/feature_contributions.csv')

print(f'Loaded {len(df)} creatures, {len(contributions_df)} contribution rows')

FileNotFoundError: [Errno 2] No such file or directory: '../../data/engineered_features.csv'

In [ ]:
# Find all (cr_tier, actual_hp) groups with 5+ creatures
groups = df.groupby(['cr_tier', 'actual_hp']).agg(
    count=('Name', 'size'),
    names=('Name', list),
    pred_min=('predicted_hp', 'min'),
    pred_max=('predicted_hp', 'max'),
    pred_range=('predicted_hp', lambda x: x.max() - x.min())
).reset_index()
groups = groups[groups['count'] >= 5].sort_values('pred_range', ascending=False)
print(f'{len(groups)} groups found\n')
groups[['cr_tier', 'actual_hp', 'count', 'pred_min', 'pred_max', 'pred_range']].round(1)

In [ ]:
# Phase 2 columns to always show
phase2_cols = [
    'ac_deviation', 'attack_deviation', 'dpr_deviation', 'save_dc_deviation',
    'has_advantage_condition', 'has_disadvantage_condition',
    'has_attackers_advantage', 'inflicts_prone'
]

# Feature flag columns (feature_*) and contribution columns (contrib_*)
feature_flag_cols = [c for c in df.columns if c.startswith('feature_') and c not in ['feature_hp', 'feature_dpr', 'feature_ac', 'feature_attack']]
contrib_cols = [c for c in contributions_df.columns if c.startswith('contrib_')]

def build_group_df(cr_tier, actual_hp):
    """Build a comparison dataframe for one HP group."""
    mask = (df['cr_tier'] == cr_tier) & (df['actual_hp'] == actual_hp)
    group = df[mask].sort_values('predicted_hp')
    names = group['Name'].tolist()
    
    # Start with core prediction columns
    core = ['Name', 'predicted_hp', 'hp_delta']
    result = group[core].copy()
    
    # Add phase 2 columns that vary
    for col in phase2_cols:
        if col in group.columns and group[col].nunique() > 1:
            result[col] = group[col].values
    
    # Add feature flags that vary within the group
    for col in feature_flag_cols:
        if col in group.columns and group[col].nunique() > 1:
            result[col.replace('feature_', 'f_')] = group[col].values
    
    # Add non-zero contribution columns that vary
    group_contribs = contributions_df[contributions_df['Name'].isin(names)]
    if len(group_contribs) > 0:
        group_contribs = group_contribs.set_index('Name').loc[names]
        for col in contrib_cols:
            if col in group_contribs.columns:
                vals = group_contribs[col]
                if vals.abs().max() > 0.5 and vals.nunique() > 1:
                    result[col] = vals.values
    
    result = result.set_index('Name')
    return result.round(1)

print(f'Ready — {len(feature_flag_cols)} feature flags, {len(contrib_cols)} contribution cols')

In [ ]:
# Generate one dataframe per group, sorted by prediction range (most spread first)
for _, row in groups.iterrows():
    tier, hp, count = row['cr_tier'], int(row['actual_hp']), int(row['count'])
    print(f'\n{"="*80}')
    print(f'{tier} | actual_hp={hp} | {count} creatures | pred range: {row["pred_range"]:.1f} HP')
    print(f'{"="*80}')
    display(build_group_df(tier, hp))